In [1]:
import numpy as np
import pandas as pd
import datetime as dt

import dask.dataframe as dd
from pathlib import Path
import re

import torch
from tqdm import tqdm
from torch import multiprocessing

In [2]:
import sys

# append the path of the
# parent directory
sys.path.append(f'..')
sys.path.append(f'../src/')
sys.path.append(f'../src/models/bat_call_detector/batdetect2/')

import batdt2_pipeline
from cfg import get_config
from pipeline import pipeline
from models.bat_call_detector.model_detector import BatCallDetector
from utils.utils import gen_empty_df

In [3]:
def apply_model(file_mapping):
    """
    Runs the batdetect2 model on a single provided audio segmens and corrects the offsets according the segment.

    Parameters
    ------------
    file_mappings : `List`
        - List of dictionaries generated by initialize_mappings()

    Returns
    ------------
    corrected_bd_dets : `pandas.DataFrame`
        - A DataFrame of detections that will also be saved in the provided output_dir under the above csv_name
        - 7 columns in this DataFrame: start_time, end_time, low_freq, high_freq, detection_confidence, event, input_file
        - Detections are always specified w.r.t their input_file; earliest start_time can be 0 and latest end_time can be 1795.
        - Events are always "Echolocation" as we are using a model that only detects search-phase calls.
    """

    bd_dets = file_mapping['model']._run_batdetect(file_mapping['audio_seg']['audio_file'])
    corrected_bd_dets = pipeline._correct_annotation_offsets(
                                                            bd_dets,
                                                            file_mapping['original_file_name'],
                                                            file_mapping['audio_seg']['offset']
                                                            )

    return corrected_bd_dets

def delete_segment(path):
    path['audio_file'].unlink(missing_ok=False)


def get_params_relevant_to_data_at_location(cfg):
    data_params = dict()
    data_params['site'] = cfg['site']
    print(f"Searching for files from {cfg['site']} in {cfg['month']} {cfg['year']}")

    hard_drive_df = dd.read_csv(f'../output_dir/ubna_data_*_collected_audio_records.csv', dtype=str).compute()
    if 'Unnamed: 0' in hard_drive_df.columns:
        hard_drive_df.drop(columns='Unnamed: 0', inplace=True)
    hard_drive_df["datetime_UTC"] = pd.DatetimeIndex(hard_drive_df["datetime_UTC"])
    hard_drive_df.set_index("datetime_UTC", inplace=True)
    
    files_from_location = filter_df_with_location(hard_drive_df, cfg)
    data_params['output_dir'] = cfg["output_dir"] / (data_params["site"].split()[0])
    print(f"Will save csv file to {data_params['output_dir']}")

    data_params['ref_audio_files'] = sorted(list(files_from_location["file_path"].apply(lambda x : Path(x)).values))
    file_status_cond = files_from_location["file_status"] == "Usable for detection"
    file_duration_cond = np.isclose(files_from_location["file_duration"].astype('float'), cfg['duration'])
    good_location_df = files_from_location.loc[file_status_cond&file_duration_cond]
    data_params['good_audio_files'] = sorted(list(good_location_df["file_path"].apply(lambda x : Path(x)).values))

    if data_params['good_audio_files'] == data_params['ref_audio_files']:
        print("All files from deployment session good!")
    else:
        print("Error files exist!")

    print(f"Will be looking at {len(data_params['good_audio_files'])} files from {data_params['site']}")

    return good_location_df, data_params


def filter_df_with_location(ubna_data_df, cfg):
    site_name_cond = ubna_data_df["site_name"] == cfg['site']
    file_year_cond = ubna_data_df.index.year == (dt.datetime.strptime(cfg['year'], '%Y')).year
    file_july_cond = ubna_data_df.index.month == (dt.datetime.strptime('July', '%B')).month
    file_august_cond = ubna_data_df.index.month == (dt.datetime.strptime('August', '%B')).month
    file_september_cond = ubna_data_df.index.month == (dt.datetime.strptime('September', '%B')).month
    file_october_cond = ubna_data_df.index.month == (dt.datetime.strptime('October', '%B')).month
    file_month_cond1 = np.logical_or(file_july_cond, file_august_cond)
    file_month_cond2 = np.logical_or(file_september_cond, file_october_cond)
    file_month_cond = np.logical_or(file_month_cond1, file_month_cond2)

    minute_cond = np.logical_or((ubna_data_df.index).minute == 30, (ubna_data_df.index).minute == 0)
    datetime_cond = np.logical_and((ubna_data_df.index).second == 0, minute_cond)
    file_error_cond = np.logical_and((ubna_data_df["file_duration"]!='File has no comment due to error!'), (ubna_data_df["file_duration"]!='File has no Audiomoth-related comment'))
    all_errors_cond = np.logical_and((ubna_data_df["file_duration"]!='Is empty!'), file_error_cond)
    file_date_cond = np.logical_and(file_year_cond, file_month_cond)

    filtered_location_df = ubna_data_df.loc[site_name_cond&datetime_cond&file_date_cond&all_errors_cond].sort_index()
    filtered_location_nightly_df = filtered_location_df.between_time(cfg['recording_start'], cfg['recording_end'], inclusive="left")

    return filtered_location_nightly_df

In [5]:
cfg = get_config()
cfg["site"] = 'Carp Pond'
cfg["year"] = '2022'
cfg["month"] = 'July'
cfg['recording_start'] = '00:00'
cfg['recording_end'] = '16:00'
cfg['duration'] = 1795
cfg["output_dir"] = Path(f'{Path.home()}/Documents/bd2_dets_20241226/output_dir')
cfg["tmp_dir"] = Path(f'{Path.home()}/Documents/bd2_dets_20241226/output')
cfg["skip_existing"] = False
cfg["num_processes"] = multiprocessing.cpu_count()

good_location_df, data_params = get_params_relevant_to_data_at_location(cfg)

Searching for files from Carp Pond in July 2022
Will save csv file to /home/exouser/Documents/bd2_dets_20241226/output_dir/Carp
Error files exist!
Will be looking at 3009 files from Carp Pond


In [7]:
f'{(len(good_location_df)*0.5)}hrs of data'

'1504.5hrs of data'

In [7]:
bd_preds = pd.DataFrame()

if not data_params['output_dir'].is_dir():
    data_params['output_dir'].mkdir(parents=True, exist_ok=True)
if not cfg['tmp_dir'].is_dir():
    cfg['tmp_dir'].mkdir(parents=True, exist_ok=True)

In [10]:
input_file = data_params['good_audio_files'][0]

In [13]:
file_path = '/'.join(input_file.parts[2:])
cleaned_path = re.sub(r"(ubna_data_\d+)_mir", r"\1", file_path)
osn_file_path = Path(f'bio230143-bucket01/{cleaned_path}')
osn_file_path

PosixPath('bio230143-bucket01/ubna_data_01/recover-20220715/UBNA_008/20220713_000000.WAV')

In [ ]:
for file in data_params['good_audio_files']:
    cfg["csv_filename"] = f"bd2__{data_params['site'].split()[0]}_{file.name.split('.')[0]}"
    if cfg['skip_existing'] & (data_params['output_dir'] / f"{cfg['csv_filename']}.csv").is_file():
        print(f'Detections for this {file.name} have already been generated!')
    else:
        print(f"Generating detections for {file.name}")
        recover_folder = good_location_df.loc[good_location_df['file_path'] == str(file), 'recover_folder'].values[0]
        audiomoth_folder = good_location_df.loc[good_location_df['file_path'] == str(file), "sd_card_num"].values[0]
        print(f"This file exists under {recover_folder}/UBNA_{audiomoth_folder}")

        packages_to_chunk = []
        chunk_instructions_and_files = dict()
        chunk_instructions_and_files['audio_file'] = file
        chunk_instructions_and_files['tmp_dir'] = cfg['tmp_dir']
        chunk_instructions_and_files['start_time'] = 0.0
        chunk_instructions_and_files['segment_duration'] = 30.0
        packages_to_chunk+=[chunk_instructions_and_files]

        if (cfg["num_processes"] <= 1):
            segmented_file_paths = []
            for package in tqdm(packages_to_chunk, desc="Segmenting Files"):
                segmented_file_paths+=[batdt2_pipeline.generate_segments_parallel(package)]
            segmented_file_paths = np.concatenate(list(segmented_file_paths))
        else:
            torch.set_num_threads(1)
            ctx = multiprocessing.get_context("spawn")
            pool = ctx.Pool(processes=cfg["num_processes"])
            segmented_file_paths = (tqdm(pool.imap(batdt2_pipeline.generate_segments_parallel, packages_to_chunk, chunksize=1), 
                            desc=f"Segmenting Files", total=len(packages_to_chunk),))
            segmented_file_paths = np.concatenate(list(segmented_file_paths))

        file_path_mappings = batdt2_pipeline.initialize_mappings(segmented_file_paths, cfg)
        if (cfg["num_processes"] <= 1):
            bd_preds = batdt2_pipeline.run_models(file_path_mappings)
        else:
            bd_preds = batdt2_pipeline.apply_models(file_path_mappings, cfg)
        bd_preds["Recover Folder"] = data_params['recover_folder']
        bd_preds["SD Card"] = data_params["audiomoth_folder"]
        bd_preds["Site name"] = data_params['site']
        batdt2_pipeline._save_predictions(bd_preds, data_params['output_dir'], cfg)
        batdt2_pipeline.delete_segments(segmented_file_paths)